# Charter Schools and Community-Wide Academic Achievement: Analysis Code

This notebook reproduces every quantitative result reported in the paper, in the order the results appear. It proceeds in three parts: a school-level comparison of charter and traditional public schools (Part 1), a geographic analysis of charter competition on nearby public schools (Part 2), and a community-level cluster analysis of charter enrollment share (Part 3).

## Setup and imports
Loads the libraries used throughout the analysis.

In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.width', None)

## Data
Loads the merged school-level dataset (978 Utah schools, 2024–25 school year) and constructs the composite proficiency outcome as the mean of the English Language Arts, mathematics, and science proficiency rates.

In [3]:
main = pd.read_excel("merged_dataset_with_teacher_retention_mapped (1) (1).xlsx",
                     sheet_name="merged_with_teacher")
main["assessment_overall"] = main[["rise_english_language_arts_proficient",
                                    "rise_mathematics_proficient",
                                    "rise_science_proficient"]].mean(axis=1)
main["avg_years_experience"] = (main["teacher_lt_3yrs"]*1.5
                                + main["teacher_4_6yrs"]*5
                                + main["teacher_7plus_yrs"]*10)
main["name_key"] = main["school_name"].str.lower().str.strip()
print("Main dataset:", main.shape)
print("Charter counts:", main["charter"].value_counts().to_dict())

Main dataset: (978, 39)
Charter counts: {0: 833, 1: 145}


## Part 1 — Descriptive statistics
Reports the mean, standard deviation, minimum, maximum, and sample size for every school-level variable used in the analysis. These values correspond to Table 1 in the paper.

In [4]:
print("="*60)
print("STEP 1 — DESCRIPTIVE STATISTICS")
print("="*60)
vars_desc = {
    "assessment_overall":"Composite proficiency",
    "rise_english_language_arts_proficient":"ELA",
    "rise_mathematics_proficient":"Math",
    "rise_science_proficient":"Science",
    "attendance_rate":"Attendance",
    "cnp_free_reduced_pct":"Free/reduced lunch",
    "English Learner_pct":"English learner",
    "Student With a Disability_pct":"Disability",
    "median_class_size":"Class size",
    "total_enrollment":"Enrollment",
    "teacher_retention_rate":"Teacher retention",
    "teacher_out_of_field":"Out-of-field",
}
for col,label in vars_desc.items():
    if col in main.columns:
        s = main[col].dropna()
        print(f"{label:<22} M={s.mean():.3f}  SD={s.std():.3f}  Min={s.min():.3f}  Max={s.max():.3f}  n={len(s)}")

STEP 1 — DESCRIPTIVE STATISTICS
Composite proficiency  M=0.259  SD=0.069  Min=0.046  Max=0.535  n=887
ELA                    M=0.281  SD=0.083  Min=0.051  Max=0.590  n=886
Math                   M=0.224  SD=0.076  Min=0.029  Max=0.577  n=880
Science                M=0.274  SD=0.069  Min=0.030  Max=0.483  n=880
Attendance             M=0.920  SD=0.035  Min=0.665  Max=1.000  n=883
Free/reduced lunch     M=0.345  SD=0.212  Min=0.036  Max=1.000  n=714
English learner        M=0.096  SD=0.117  Min=0.000  Max=0.906  n=858
Disability             M=0.150  SD=0.085  Min=0.000  Max=1.000  n=858
Class size             M=23.579  SD=5.544  Min=1.000  Max=38.000  n=906
Enrollment             M=681.862  SD=543.485  Min=0.000  Max=4282.000  n=864
Teacher retention      M=0.587  SD=0.141  Min=0.013  Max=0.930  n=840
Out-of-field           M=0.085  SD=0.084  Min=0.000  Max=0.857  n=846


## Part 1 — Charter versus public means
Compares raw average proficiency between charter and traditional public schools, overall and within each tested subject, before any statistical controls are applied.

In [5]:
print("="*60)
print("STEP 1 — CHARTER VS PUBLIC MEANS")
print("="*60)
pub = main[main["charter"]==0]; cha = main[main["charter"]==1]
for col,label in [("assessment_overall","Overall"),
                  ("rise_english_language_arts_proficient","ELA"),
                  ("rise_mathematics_proficient","Math"),
                  ("rise_science_proficient","Science")]:
    print(f"{label:<10} Public={pub[col].mean():.3f}  Charter={cha[col].mean():.3f}")

print("\n" + "="*60)
print("STEP 1 — CORRELATIONS WITH PROFICIENCY")
print("="*60)
corr_cols = ["assessment_overall","attendance_rate","cnp_free_reduced_pct",
             "Economically Disadvantaged_pct","English Learner_pct",
             "Student With a Disability_pct","Homeless_pct",
             "median_class_size","total_enrollment","charter",
             "teacher_retention_rate","avg_years_experience","teacher_out_of_field"]
corr_cols = [c for c in corr_cols if c in main.columns]
corr = main[corr_cols].corr()["assessment_overall"].drop("assessment_overall").sort_values()
print(corr.round(3).to_string())

STEP 1 — CHARTER VS PUBLIC MEANS
Overall    Public=0.259  Charter=0.263
ELA        Public=0.279  Charter=0.291
Math       Public=0.224  Charter=0.221
Science    Public=0.273  Charter=0.280

STEP 1 — CORRELATIONS WITH PROFICIENCY
cnp_free_reduced_pct             -0.723
English Learner_pct              -0.638
Economically Disadvantaged_pct   -0.630
Student With a Disability_pct    -0.318
Homeless_pct                     -0.302
teacher_out_of_field             -0.078
charter                           0.023
total_enrollment                  0.229
median_class_size                 0.234
avg_years_experience              0.249
teacher_retention_rate            0.249
attendance_rate                   0.446


## Part 1 — Regression
Estimates the association between charter status and proficiency after adjusting for the baseline controls, using ridge regression with standardized predictors and five-fold cross-validation. Also reports the ordinary least squares significance test for the charter coefficient and the change in adjusted R² when charter status is added to the model.

In [6]:
print("="*60)
print("STEP 1 — REGRESSION")
print("="*60)
features = ["attendance_rate","cnp_free_reduced_pct","English Learner_pct",
            "Student With a Disability_pct","charter","median_class_size",
            "total_enrollment","teacher_retention_rate","avg_years_experience",
            "teacher_out_of_field"]
features = [f for f in features if f in main.columns]
sub = main[features+["assessment_overall"]].dropna()
X = sub[features]; y = sub["assessment_overall"]
imp = SimpleImputer(strategy="median")
Xi = imp.fit_transform(X)
Xs = StandardScaler().fit_transform(Xi)

ridge = Ridge(alpha=1.0).fit(Xs,y)
cv = cross_val_score(ridge, Xs, y, cv=5, scoring="r2")
print(f"Ridge R²={ridge.score(Xs,y):.4f}  CV R²={cv.mean():.4f}")
print("\nRidge standardized coefficients:")
for f,c in zip(features, ridge.coef_):
    print(f"  {f:<32} {c:+.4f}")

ols = sm.OLS(y, sm.add_constant(Xs)).fit()
print(f"\nOLS R²={ols.rsquared:.4f}  Adj R²={ols.rsquared_adj:.4f}")
print("OLS coefficients and p-values:")
for f,c,p in zip(features, ols.params[1:], ols.pvalues[1:]):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f"  {f:<32} coef={c:+.4f}  p={p:.4f} {sig}")

# Adjusted R2 with and without charter
def adj_r2(r2,n,k): return 1-(1-r2)*(n-1)/(n-k-1)
feats_no = [f for f in features if f!="charter"]
for label,fl in [("WITHOUT charter",feats_no),("WITH charter",features)]:
    s2 = main[fl+["assessment_overall"]].dropna()
    Xi2 = imp.fit_transform(s2[fl]); Xs2 = StandardScaler().fit_transform(Xi2)
    m2 = Ridge(alpha=1.0).fit(Xs2, s2["assessment_overall"])
    r2 = m2.score(Xs2, s2["assessment_overall"])
    print(f"{label}: R²={r2:.4f}  Adj R²={adj_r2(r2,len(s2),len(fl)):.4f}")

STEP 1 — REGRESSION
Ridge R²=0.6612  CV R²=0.5821

Ridge standardized coefficients:
  attendance_rate                  +0.0105
  cnp_free_reduced_pct             -0.0174
  English Learner_pct              -0.0296
  Student With a Disability_pct    -0.0160
  charter                          +0.0042
  median_class_size                +0.0001
  total_enrollment                 +0.0016
  teacher_retention_rate           +0.0045
  avg_years_experience             +0.0046
  teacher_out_of_field             +0.0038

OLS R²=0.6612  Adj R²=0.6551
OLS coefficients and p-values:
  attendance_rate                  coef=+0.0105  p=0.0000 ***
  cnp_free_reduced_pct             coef=-0.0174  p=0.0000 ***
  English Learner_pct              coef=-0.0297  p=0.0000 ***
  Student With a Disability_pct    coef=-0.0160  p=0.0000 ***
  charter                          coef=+0.0042  p=0.0443 *
  median_class_size                coef=+0.0000  p=0.9892 ns
  total_enrollment                 coef=+0.0016  p=0.484

## Parts 2 and 3 — Geographic setup
Merges the geocoded school locations (latitude and longitude) obtained from the National Center for Education Statistics onto the analysis dataset. These coordinates support the distance-based measures used in the competition and cluster analyses.

In [8]:
print("="*60)
print("STEP 2/3 SETUP — GEOCODED SCHOOL DATA")
print("="*60)
nces = pd.read_csv("ncesdata_AF584DD8_geocoded.csv")
nces = nces[nces["Latitude"].notna() & nces["Longitude"].notna()].copy()
nces["Students"] = pd.to_numeric(nces["Students"], errors="coerce").fillna(0)
nces["name_key"] = nces["School Name"].str.lower().str.strip()

order = {'PK':-1,'KG':0,'01':1,'02':2,'03':3,'04':4,'05':5,'06':6,
         '07':7,'08':8,'09':9,'10':10,'11':11,'12':12}
def gnum(g): return order.get(str(g).strip(), None)

def is_high_school(low, high):
    lo,hi = gnum(low),gnum(high)
    if lo is None or hi is None: return False
    return hi >= 11 and lo >= 7

def is_middle_or_junior(low, high):
    lo,hi = gnum(low),gnum(high)
    if lo is None or hi is None: return False
    return lo >= 5 and hi >= 6 and hi <= 9

def is_elementary(low, high):
    lo,hi = gnum(low),gnum(high)
    if lo is None or hi is None: return False
    return lo <= 3 and hi <= 6

def serves_elementary(low, high):
    lo,hi = gnum(low),gnum(high)
    if lo is None or hi is None: return False
    return lo <= 5 and hi <= 8

nces["is_hs"]       = nces.apply(lambda r: is_high_school(r["Low Grade"], r["High Grade"]), axis=1)
nces["is_mid_jr"]   = nces.apply(lambda r: is_middle_or_junior(r["Low Grade"], r["High Grade"]), axis=1)
nces["is_elem"]     = nces.apply(lambda r: is_elementary(r["Low Grade"], r["High Grade"]), axis=1)
nces["serves_elem"] = nces.apply(lambda r: serves_elementary(r["Low Grade"], r["High Grade"]), axis=1)

nces_slim = nces[["name_key","Latitude","Longitude","Charter","Students",
                  "is_hs","is_mid_jr","is_elem","serves_elem"]].drop_duplicates("name_key")

geo = main.merge(nces_slim, on="name_key", how="left")
geo = geo[geo["Latitude"].notna() & geo["assessment_overall"].notna()].copy()
geo["is_charter"] = (geo["Charter"]=="Yes").astype(int)
geo["enrollment"] = pd.to_numeric(geo["Students"], errors="coerce").fillna(geo["total_enrollment"]).fillna(100)

print("Schools with coordinates + proficiency:", len(geo))
print("  Charter:", geo["is_charter"].sum(), " Public:", (geo["is_charter"]==0).sum())
print("  High schools:", geo["is_hs"].sum())
print("  Charter middle-only schools:", ((geo["is_charter"]==1)&(geo["is_mid_jr"])&(~geo["is_hs"])).sum())

STEP 2/3 SETUP — GEOCODED SCHOOL DATA
Schools with coordinates + proficiency: 857
  Charter: 122  Public: 735
  High schools: 126
  Charter middle-only schools: 1


## Distance function
Defines the haversine formula, which computes the straight-line distance in miles between two schools from their latitude and longitude.

In [9]:
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1,lon1,lat2,lon2 = map(radians,[lat1,lon1,lat2,lon2])
    dlat,dlon = lat2-lat1, lon2-lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R*2*atan2(sqrt(a), sqrt(1-a))

def wavg(values, weights):
    vals, wts = [], []
    for v,w in zip(values,weights):
        if pd.notna(v) and pd.notna(w) and w>0:
            vals.append(v); wts.append(w)
    return np.average(vals, weights=wts) if vals else np.nan

print("Helper functions defined.")

Helper functions defined.


## Part 3 — Cluster analysis (high-school-anchored)
Groups schools into geographic clusters, each anchored on a traditional public high school with every charter and public middle school assigned to its nearest anchor. Each cluster's variables are computed as enrollment-weighted averages, and cluster proficiency is regressed on charter enrollment share with the baseline controls.

In [10]:
print("="*60)
print("STEP 2a — HIGH-SCHOOL-CENTERED AREA ANALYSIS")
print("="*60)

centers = geo[(geo["is_charter"]==0) & (geo["is_hs"]==True)].copy().reset_index(drop=True)
charters = geo[geo["is_charter"]==1].copy()
pub_mid  = geo[(geo["is_charter"]==0) & (geo["is_mid_jr"]==True)].copy()
assignable = pd.concat([charters, pub_mid]).reset_index(drop=True)
print(f"Centers (public high schools): {len(centers)}")
print(f"Assignable (charters + public mid/jr): {len(assignable)}")

# Assign each assignable school to geographically closest public HS
assignments = {i: [] for i in range(len(centers))}
for _, s in assignable.iterrows():
    dists = centers.apply(lambda c: haversine_miles(s["Latitude"],s["Longitude"],
                                                    c["Latitude"],c["Longitude"]), axis=1)
    assignments[dists.idxmin()].append(s)

wvars = {
    "cnp_free_reduced_pct":"W_Poverty",
    "English Learner_pct":"W_EL",
    "attendance_rate":"W_Attendance",
    "Student With a Disability_pct":"W_Disability",
    "median_class_size":"W_ClassSize",
    "teacher_retention_rate":"W_TeacherRetention",
    "avg_years_experience":"W_TeacherExp",
    "teacher_out_of_field":"W_OutOfField",
}

rows = []
for idx, center in centers.iterrows():
    members = assignments[idx]
    school_list = [center] + members
    enr   = [s["enrollment"] for s in school_list]
    is_ch = [s["is_charter"] for s in school_list]
    total_enr = sum(enr)
    charter_enr = sum(e for e,c in zip(enr,is_ch) if c==1)
    row = {
        "Public_HS_Center": center["school_name"],
        "Total_Area_Enr": int(total_enr),
        "N_Charters": sum(is_ch),
        "Charter_Enr_Share": charter_enr/total_enr if total_enr>0 else 0,
        "W_Proficiency": wavg([s["assessment_overall"] for s in school_list], enr),
    }
    for col,label in wvars.items():
        row[label] = wavg([s[col] for s in school_list], enr)
    rows.append(row)

areas = pd.DataFrame(rows)
areas.to_csv("hs_areas.csv", index=False)
print(f"Total areas: {len(areas)}  | with >=1 charter: {(areas['N_Charters']>0).sum()}")

all_features = ["Charter_Enr_Share","W_Poverty","W_EL","W_Attendance","W_Disability",
                "W_ClassSize","W_TeacherRetention","W_TeacherExp","W_OutOfField"]

def run_area_reg(data, label):
    reg = data[all_features+["W_Proficiency"]].dropna()
    X = reg[all_features].values; y = reg["W_Proficiency"].values
    Xi = SimpleImputer(strategy="median").fit_transform(X)
    Xs = StandardScaler().fit_transform(Xi)
    ols = sm.OLS(y, sm.add_constant(Xs)).fit()
    bivar = pd.Series(reg["Charter_Enr_Share"].values).corr(pd.Series(y))
    print(f"\n--- {label} ---")
    print(f"N areas={len(reg)}  bivariate r={bivar:.4f}  R²={ols.rsquared:.4f}  Adj R²={ols.rsquared_adj:.4f}")
    print(f"  {'Charter_Enr_Share':<22} coef={ols.params[1]:+.5f}  p={ols.pvalues[1]:.4f}")
    for f,c,p in zip(all_features[1:], ols.params[2:], ols.pvalues[2:]):
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
        print(f"  {f:<22} coef={c:+.5f}  p={p:.4f} {sig}")

run_area_reg(areas, "ALL high-school areas (full controls)")
run_area_reg(areas[areas["N_Charters"]>0], "ONLY areas with charters (full controls)")

STEP 2a — HIGH-SCHOOL-CENTERED AREA ANALYSIS
Centers (public high schools): 106
Assignable (charters + public mid/jr): 254
Total areas: 106  | with >=1 charter: 58

--- ALL high-school areas (full controls) ---
N areas=100  bivariate r=-0.2476  R²=0.7612  Adj R²=0.7373
  Charter_Enr_Share      coef=+0.00360  p=0.3347
  W_Poverty              coef=-0.00730  p=0.1992 ns
  W_EL                   coef=-0.02848  p=0.0000 ***
  W_Attendance           coef=+0.01773  p=0.0000 ***
  W_Disability           coef=-0.01163  p=0.0029 **
  W_ClassSize            coef=-0.00165  p=0.7269 ns
  W_TeacherRetention     coef=+0.00070  p=0.8997 ns
  W_TeacherExp           coef=+0.01533  p=0.0150 *
  W_OutOfField           coef=-0.00243  p=0.6252 ns

--- ONLY areas with charters (full controls) ---
N areas=56  bivariate r=-0.3449  R²=0.8078  Adj R²=0.7702
  Charter_Enr_Share      coef=-0.00074  p=0.8781
  W_Poverty              coef=-0.00199  p=0.7655 ns
  W_EL                   coef=-0.03177  p=0.0000 ***
  

## Part 3 — Grid-shifting robustness check
Repeats the community-level analysis using a 10-by-10 mile grid rather than high-school-anchored clusters, shifting the grid position across configurations to confirm that the results do not depend on where the cluster boundaries are drawn. The check is run separately for high schools and for elementary schools.

In [11]:
print("="*60)
print("STEP 2b — GRID-SHIFTING ROBUSTNESS")
print("="*60)

UTAH_LAT = 39.5
MILES_PER_LAT = 69.0
MILES_PER_LON = 69.0 * cos(radians(UTAH_LAT))
GRID_SIZE = 10
SHIFT_STEP = 2
N_SHIFTS = 5

grid_wvars = ["cnp_free_reduced_pct","English Learner_pct","attendance_rate",
              "Student With a Disability_pct","median_class_size",
              "teacher_retention_rate","avg_years_experience","teacher_out_of_field"]

def grid_shift_analysis(school_subset, label, min_schools=3):
    results = []
    for sx in range(N_SHIFTS):
        for sy in range(N_SHIFTS):
            shift_lon = (sx*SHIFT_STEP)/MILES_PER_LON
            shift_lat = (sy*SHIFT_STEP)/MILES_PER_LAT
            df = school_subset.copy()
            df["gx"] = (((df["Longitude"]-shift_lon)-df["Longitude"].min())*MILES_PER_LON/GRID_SIZE).astype(int)
            df["gy"] = (((df["Latitude"]-shift_lat)-df["Latitude"].min())*MILES_PER_LAT/GRID_SIZE).astype(int)
            df["gid"] = df["gx"].astype(str)+"_"+df["gy"].astype(str)

            def agg(g):
                enr = g["enrollment"].sum()
                ce = g.loc[g["is_charter"]==1,"enrollment"].sum()
                d = {"n":len(g),
                     "charter_share": ce/enr if enr>0 else np.nan,
                     "prof": np.average(g["assessment_overall"], weights=g["enrollment"])}
                for v in grid_wvars:
                    vv = g[v].dropna(); ww = g.loc[g[v].notna(),"enrollment"]
                    d[v] = np.average(vv, weights=ww) if len(vv)>0 and ww.sum()>0 else np.nan
                return pd.Series(d)

            grid = df.groupby("gid").apply(agg).reset_index()
            gf = grid[grid["n"]>=min_schools].dropna(subset=["charter_share","prof"]+grid_wvars)
            if len(gf) < 10: continue
            feats = ["charter_share"]+grid_wvars
            X = gf[feats].values; y = gf["prof"].values
            Xi = SimpleImputer(strategy="median").fit_transform(X)
            Xs = StandardScaler().fit_transform(Xi)
            ols = sm.OLS(y, sm.add_constant(Xs)).fit()
            results.append({
                "charter_coef": ols.params[1], "charter_p": ols.pvalues[1],
                "significant": ols.pvalues[1]<0.05,
                "bivar_r": pd.Series(gf["charter_share"].values).corr(pd.Series(y)),
                "n_grids": len(gf),
            })
    r = pd.DataFrame(results)
    print(f"\n--- {label} ---")
    print(f"Configurations: {len(r)}  | mean grids/config: {r['n_grids'].mean():.0f}")
    print(f"Negative coef: {(r['charter_coef']<0).sum()}/{len(r)}  | significant: {r['significant'].sum()}/{len(r)}")
    print(f"Mean charter coef: {r['charter_coef'].mean():.5f}  | mean p: {r['charter_p'].mean():.4f}  | mean bivar r: {r['bivar_r'].mean():.4f}")
    return r

# High-school-serving schools
hs_serving = geo[geo["is_hs"]==True].copy()
grid_shift_analysis(hs_serving, "HIGH SCHOOLS (10x10 mi)")

# Elementary-serving schools
elem_serving = geo[geo["serves_elem"]==True].copy()
grid_shift_analysis(elem_serving, "ELEMENTARY (10x10 mi)")

STEP 2b — GRID-SHIFTING ROBUSTNESS

--- HIGH SCHOOLS (10x10 mi) ---
Configurations: 25  | mean grids/config: 13
Negative coef: 16/25  | significant: 2/25
Mean charter coef: -0.01059  | mean p: 0.4234  | mean bivar r: -0.1849

--- ELEMENTARY (10x10 mi) ---
Configurations: 25  | mean grids/config: 33
Negative coef: 14/25  | significant: 0/25
Mean charter coef: -0.00049  | mean p: 0.4328  | mean bivar r: -0.1435


,charter_coef,charter_p,significant,bivar_r,n_grids
0,-0.003195,0.472482,False,-0.161062,31
1,-0.004933,0.208417,False,-0.311093,34
2,0.002041,0.626174,False,-0.064176,34
3,0.000736,0.871523,False,-0.107545,31
4,0.001963,0.687565,False,0.050199,31
5,-0.006568,0.302254,False,-0.319286,32
6,-0.004563,0.307209,False,-0.292387,31
7,-0.002224,0.656251,False,-0.196217,32
8,-0.004178,0.471552,False,-0.354931,32
9,0.003831,0.430825,False,-0.115081,33


## Part 2 — Charter competition score
Constructs a competition score for each traditional public school by summing the enrollment of each grade-overlapping charter school divided by its distance, then regresses public school proficiency on this score under two constructions of the measure. This corresponds to the competition analysis and Figure 2 in the paper.

In [12]:
print("="*60)
print("STEP 3 — CHARTER COMPETITION SCORE (public schools)")
print("="*60)

# School-level classification for matching: a charter competes with a public
# school if they overlap in level. We use serves_elem / is_mid_jr / is_hs.
def levels_of(row):
    L = set()
    if row["serves_elem"]: L.add("elem")
    if row["is_mid_jr"]:   L.add("mid")
    if row["is_hs"]:       L.add("high")
    return L

geo["level_set"] = geo.apply(levels_of, axis=1)

public_geo  = geo[geo["is_charter"]==0].copy().reset_index(drop=True)
charter_geo = geo[geo["is_charter"]==1].copy().reset_index(drop=True)

def compute_competition(radius=None, scale_by_enrollment=False):
    scores = []
    for _, pub in public_geo.iterrows():
        score = 0.0
        pub_levels = pub["level_set"]
        for _, cha in charter_geo.iterrows():
            if pub_levels & cha["level_set"]:
                d = haversine_miles(pub["Latitude"],pub["Longitude"],
                                    cha["Latitude"],cha["Longitude"])
                if d > 0 and (radius is None or d <= radius):
                    score += cha["enrollment"] / d
        if scale_by_enrollment and pub["enrollment"]>0:
            score = score / pub["enrollment"]
        scores.append(score)
    return scores

public_geo["comp_v1"] = compute_competition(radius=None, scale_by_enrollment=False)
public_geo["comp_v2"] = compute_competition(radius=10, scale_by_enrollment=True)

comp_features_base = ["attendance_rate","cnp_free_reduced_pct","English Learner_pct",
                      "Student With a Disability_pct","median_class_size","total_enrollment",
                      "teacher_retention_rate","avg_years_experience","teacher_out_of_field"]

def run_competition_reg(compvar, label):
    feats = comp_features_base + [compvar]
    reg = public_geo[feats+["assessment_overall"]].dropna()
    X = reg[feats].values; y = reg["assessment_overall"].values
    Xi = SimpleImputer(strategy="median").fit_transform(X)
    Xs = StandardScaler().fit_transform(Xi)
    ols = sm.OLS(y, sm.add_constant(Xs)).fit()
    print(f"\n--- {label} ---  (N={len(reg)})")
    print(f"R²={ols.rsquared:.4f}  Adj R²={ols.rsquared_adj:.4f}")
    ci = feats.index(compvar)
    print(f"  {compvar:<22} coef={ols.params[1+ci]:+.5f}  p={ols.pvalues[1+ci]:.4f}")
    bivar = pd.Series(reg[compvar].values).corr(pd.Series(y))
    print(f"  bivariate r ({compvar} vs proficiency) = {bivar:.4f}")

run_competition_reg("comp_v1", "Competition v1 (no radius, unscaled)")
run_competition_reg("comp_v2", "Competition v2 (10-mile radius, enrollment-scaled)")

STEP 3 — CHARTER COMPETITION SCORE (public schools)

--- Competition v1 (no radius, unscaled) ---  (N=494)
R²=0.6817  Adj R²=0.6752
  comp_v1                coef=-0.01005  p=0.0000
  bivariate r (comp_v1 vs proficiency) = -0.3527

--- Competition v2 (10-mile radius, enrollment-scaled) ---  (N=494)
R²=0.6738  Adj R²=0.6671
  comp_v2                coef=-0.00600  p=0.0097
  bivariate r (comp_v2 vs proficiency) = -0.3852
